<a href="https://colab.research.google.com/github/edermartelinho/credit-score-mlops/blob/main/credit_score_mlops1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Instalação de Dependencia

In [18]:
!pip install -q fastapi uvicorn pydantic xgboost scikit-learn joblib mlflow pytest
print("Dependencias instaladas com sucesso!")

Dependencias instaladas com sucesso!


#Estrutura de pastas

In [19]:
import os

# Lista de diretórios que compõem o repositório
directories = [
    "credit-score-mlops/app",
    "credit-score-mlops/src",
    "credit-score-mlops/data/raw",
    "credit-score-mlops/data/processed",
    "credit-score-mlops/models",
    "credit-score-mlops/tests"
]

for d in directories:
    os.makedirs(d, exist_ok=True)

print("Estrutura de diretórios criada:")
!tree credit-score-mlops || find credit-score-mlops -type d

Estrutura de diretórios criada:
/bin/bash: line 1: tree: command not found
credit-score-mlops
credit-score-mlops/tests
credit-score-mlops/data
credit-score-mlops/data/raw
credit-score-mlops/data/processed
credit-score-mlops/models
credit-score-mlops/app
credit-score-mlops/src


#Arquivos de configuracao
##Utilizamos o comando  %%writefile para criar os arquivos diretamente no sistema de arquivos

In [20]:
%%writefile credit-score-mlops/pytest.ini
[pytest]
pythonpath = .

Writing credit-score-mlops/pytest.ini


In [21]:
%%writefile credit-score-mlops/requirements.txt
fastapi>=0.100.0
uvicorn>=0.22.0
pydantic>=2.0.0
scikit-learn>=1.3.0
xgboost>=1.7.0
joblib>=1.3.0
pandas>=2.0.0
mlflow>=2.5.0
pytest>=7.4.0
streamlit>=1.25.0

Writing credit-score-mlops/requirements.txt


In [22]:
%%writefile credit-score-mlops/.gitignore
.venv/
venv/
__pycache__/
*.pyc
.pytest_cache/
.idea/
mlruns/
models/*.joblib
data/raw/*
data/processed/*

Writing credit-score-mlops/.gitignore


#Contrato da Api

In [23]:
%%writefile credit-score-mlops/src/schemas.py
from pydantic import BaseModel, Field

class CreditApplicationRequest(BaseModel):
    idade: int = Field(..., ge=18, le=100, description="Idade em anos", example=35)
    renda_mensal: float = Field(..., gt=0, description="Renda mensal em R$", example=6500.0)
    valor_emprestimo: float = Field(..., gt=0, description="Valor solicitado", example=20000.0)
    num_parcelas: int = Field(..., ge=1, le=84, description="Quantidade de parcelas", example=36)
    taxa_comprometimento_renda: float = Field(..., ge=0.0, le=1.0, description="Comprometimento de renda", example=0.25)
    historico_inadimplencia: int = Field(..., ge=0, le=1, description="Histórico de inadimplência (0 ou 1)", example=0)
    num_consultas_spc: int = Field(..., ge=0, description="Consultas SPC/Serasa", example=1)

class CreditPredictionResponse(BaseModel):
    score: int
    probabilidade_inadimplencia: float
    decisao: str
    classe_risco: str
    tempo_inferencia_ms: float

Writing credit-score-mlops/src/schemas.py


#Script de Treinamento com Scikit-Learn / XGBoost
##Este script gera um conjunto de dados sintético, treina um pipeline com pré-processamento (StandardScaler) e modelo (RandomForestClassifier), e salva os artefatos na pasta models/

In [24]:
%%writefile credit-score-mlops/src/train.py
import os
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

def generate_synthetic_data(n_samples: int = 1000):
    np.random.seed(42)
    idade = np.random.randint(18, 70, n_samples)
    renda_mensal = np.random.uniform(1500, 20000, n_samples)
    valor_emprestimo = np.random.uniform(2000, 50000, n_samples)
    num_parcelas = np.random.randint(6, 60, n_samples)
    taxa_comprometimento_renda = np.random.uniform(0.05, 0.6, n_samples)
    historico_inadimplencia = np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
    num_consultas_spc = np.random.randint(0, 8, n_samples)

    # Regra sintética de inadimplência
    score_raw = (renda_mensal * 0.0005) - (taxa_comprometimento_renda * 5) - (historico_inadimplencia * 3) - (num_consultas_spc * 0.5)
    prob = 1 / (1 + np.exp(score_raw))
    target = (prob > 0.5).astype(int)

    df = pd.DataFrame({
        "idade": idade,
        "renda_mensal": renda_mensal,
        "valor_emprestimo": valor_emprestimo,
        "num_parcelas": num_parcelas,
        "taxa_comprometimento_renda": taxa_comprometimento_renda,
        "historico_inadimplencia": historico_inadimplencia,
        "num_consultas_spc": num_consultas_spc,
        "target": target
    })
    return df

def run_training():
    print("🔄 Gerando dados e iniciando treinamento...")
    df = generate_synthetic_data()

    X = df.drop(columns=["target"])
    y = df["target"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    print(f"📊 ROC-AUC do Modelo: {auc:.4f}")

    # Salvar Artefatos
    models_dir = Path("credit-score-mlops/models")
    models_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, models_dir / "credit_model.joblib")
    joblib.dump(scaler, models_dir / "preprocessor.joblib")
    print("✅ Artefatos salvos com sucesso em credit-score-mlops/models/")

if __name__ == "__main__":
    run_training()

Writing credit-score-mlops/src/train.py


#Classe de Inferencia

In [25]:
%%writefile credit-score-mlops/src/predict.py
import time
from pathlib import Path
import joblib
import pandas as pd
from src.schemas import CreditApplicationRequest, CreditPredictionResponse

class CreditPredictor:
    def __init__(self, model_dir: str = "credit-score-mlops/models"):
        self.model_dir = Path(model_dir)
        self.model = joblib.load(self.model_dir / "credit_model.joblib")
        self.preprocessor = joblib.load(self.model_dir / "preprocessor.joblib")

    def predict(self, request_data: CreditApplicationRequest) -> CreditPredictionResponse:
        start_time = time.perf_counter()

        input_df = pd.DataFrame([request_data.model_dump()])
        processed_data = self.preprocessor.transform(input_df)

        proba_inadimplencia = float(self.model.predict_proba(processed_data)[0][1])
        score = int((1.0 - proba_inadimplencia) * 1000)

        if score >= 750:
            decisao, classe_risco = "APROVADO", "BAIXO"
        elif 500 <= score < 750:
            decisao, classe_risco = "ANALISE_MANUAL", "MEDIO"
        else:
            decisao, classe_risco = "RECUSADO", "ALTO"

        execution_time_ms = round((time.perf_counter() - start_time) * 1000, 2)

        return CreditPredictionResponse(
            score=score,
            probabilidade_inadimplencia=round(proba_inadimplencia, 4),
            decisao=decisao,
            classe_risco=classe_risco,
            tempo_inferencia_ms=execution_time_ms
        )

Writing credit-score-mlops/src/predict.py


#Testes Unitario e de Validação

In [26]:
%%writefile credit-score-mlops/tests/test_predict.py
from src.schemas import CreditApplicationRequest
from src.predict import CreditPredictor

def test_credit_prediction():
    predictor = CreditPredictor(model_dir="credit-score-mlops/models")
    payload = CreditApplicationRequest(
        idade=35,
        renda_mensal=8000.0,
        valor_emprestimo=10000.0,
        num_parcelas=24,
        taxa_comprometimento_renda=0.15,
        historico_inadimplencia=0,
        num_consultas_spc=0
    )
    response = predictor.predict(payload)
    assert 0 <= response.score <= 1000
    assert response.decisao in ["APROVADO", "ANALISE_MANUAL", "RECUSADO"]

Writing credit-score-mlops/tests/test_predict.py


#Rodamos o script de treinamento e executamos o Pytest

In [27]:
# 1. Executa o treinamento para gerar os modelos na pasta
!python credit-score-mlops/src/train.py

# 2. Executa os testes unitários com Pytest
!pytest credit-score-mlops/tests/

🔄 Gerando dados e iniciando treinamento...
📊 ROC-AUC do Modelo: 0.9822
✅ Artefatos salvos com sucesso em credit-score-mlops/models/
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/credit-score-mlops/credit-score-mlops
configfile: pytest.ini
plugins: langsmith-0.10.2, anyio-4.14.2, typeguard-4.5.2
collected 1 item                                                               

credit-score-mlops/tests/test_predict.py .                               [100%]

============================== 1 passed in 4.51s ===============================


#Servidor FastApi

In [28]:
%%writefile credit-score-mlops/app/main.py
from fastapi import FastAPI
from src.schemas import CreditApplicationRequest, CreditPredictionResponse
from src.predict import CreditPredictor

app = FastAPI(title="Credit Score MLOps API")
predictor = CreditPredictor(model_dir="models")

@app.get("/health")
def health_check():
    return {"status": "ok"}

@app.post("/predict", response_model=CreditPredictionResponse)
def predict(request: CreditApplicationRequest):
    return predictor.predict(request)

Writing credit-score-mlops/app/main.py
